# Options Recipes Example

This notebook demonstrates the recipes for fitting forwards curves and matching them to options orderbook data.


In [1]:
%load_ext autoreload
%autoreload 2

from datetime import datetime, timedelta, date
from functools import partial
import numpy as np
import polars as pl

from okx.store import OrderbookStore
from okx.recipes.options import prepare_options, build_forwards_options_comparison
from okx.recipes.forwards import build_forwards_pchip, build_forwards_kalman, prepare_pillars

In [2]:
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite",
    batch_days=5
)

In [30]:
store.clear_cache()

Cleared all caches


In [3]:
start_date = date(2025, 9, 1)
end_date = date(2025, 9, 2)
dates = [start_date + timedelta(days=i) for i in range((end_date - start_date).days + 1)]

print(f"Testing with dates: {dates[0]} to {dates[-1]}")

# just one date for now
dates = [date(2025, 9, 2)]


Testing with dates: 2025-09-01 to 2025-09-02


In [21]:
options_df = prepare_options(
    store,
    inst_family='BTC-USD',
    dates=dates,
    forwards_recipe=build_forwards_kalman,
    binning='5m'
)

Time taken to fetch options: 0:00:00.007305
Number of null entries in bid_1_px: 8415
Number of null entries in ask_1_px: 6625
Number of null entries in bid_1_px and ask_1_px: 45
Number of negative entries in bid_1_px: 0
Number of negative entries in ask_1_px: 0
Number of bad options (ask == 0): 0
Number of bad options (bid == 0): 0
Number of bad options (ask == 0 and bid == 0): 0
Number of bad options (ask < bid): 0
Number of bad options (ask < bid and ask == 0): 0
Number of bad options (ask < bid and ask != 0): 0
Total number of entries in df_options: 149847
Time taken to fetch forwards: 0:00:00.003963
Time taken to build forward lookup: 0:00:00.001011


Matching forwards to options: 100%|██████████| 287/287 [00:00<00:00, 11467.49it/s]

Time taken to match forwards to options: core 0:00:00.035144, incl. setup 0:00:00.035152 (matched 149,847 / 149,847 options)


In [13]:
print(options_df.shape)
print(options_df.head())
options_df_filtered = options_df.filter((pl.col('bid_1_px').is_not_null()) & (pl.col('ask_1_px').is_not_null()))
print(options_df_filtered.shape)
print(options_df_filtered.head())
duplicate_counts = (
    options_df_filtered
    .group_by([col for col in options_df_filtered.columns])
    .count()
    .filter(pl.col("count") > 1)
    .sort("count", descending=True)
)

print("Duplicate rows and their counts:")
print(duplicate_counts)


(29216457, 11)
shape: (5, 11)
┌────────────┬────────────┬──────────┬──────────┬───┬──────────┬───────────┬───────────┬───────────┐
│ timeMs     ┆ symbol     ┆ bid_1_px ┆ ask_1_px ┆ … ┆ opt_type ┆ F_bid     ┆ F_ask     ┆ moneyness │
│ ---        ┆ ---        ┆ ---      ┆ ---      ┆   ┆ ---      ┆ ---       ┆ ---       ┆ ---       │
│ i64        ┆ str        ┆ f64      ┆ f64      ┆   ┆ str      ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪════════════╪══════════╪══════════╪═══╪══════════╪═══════════╪═══════════╪═══════════╡
│ 1756771507 ┆ BTC-USD-25 ┆ 0.005    ┆ null     ┆ … ┆ C        ┆ 109120.72 ┆ 109120.82 ┆ 0.916416  │
│ 999        ┆ 0903-10000 ┆          ┆          ┆   ┆          ┆ 9946      ┆ 9971      ┆           │
│            ┆ 0-C.OK     ┆          ┆          ┆   ┆          ┆           ┆           ┆           │
│ 1756771519 ┆ BTC-USD-25 ┆ 0.005    ┆ null     ┆ … ┆ C        ┆ 109120.72 ┆ 109120.82 ┆ 0.916416  │
│ 307        ┆ 0903-10000 ┆          ┆          ┆   ┆        

/var/folders/fl/fdwpgpsx7fx15p__07t93bhr0000gn/T/ipykernel_28342/3014190339.py:9: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()


Duplicate rows and their counts:
shape: (0, 12)
┌────────┬────────┬──────────┬──────────┬───┬───────┬───────┬───────────┬───────┐
│ timeMs ┆ symbol ┆ bid_1_px ┆ ask_1_px ┆ … ┆ F_bid ┆ F_ask ┆ moneyness ┆ count │
│ ---    ┆ ---    ┆ ---      ┆ ---      ┆   ┆ ---   ┆ ---   ┆ ---       ┆ ---   │
│ i64    ┆ str    ┆ f64      ┆ f64      ┆   ┆ f64   ┆ f64   ┆ f64       ┆ u32   │
╞════════╪════════╪══════════╪══════════╪═══╪═══════╪═══════╪═══════════╪═══════╡
└────────┴────────┴──────────┴──────────┴───┴───────┴───────┴───────────┴───────┘


In [14]:
def show_df_time_range(name, df):
    if df.is_empty():
        print(f"{name}: DataFrame is empty")
        return
    timeMs = df['timeMs'].to_numpy()
    earliest = timeMs.min()
    latest = timeMs.max()
    if hasattr(earliest, 'item'):
        earliest = earliest.item()
    if hasattr(latest, 'item'):
        latest = latest.item()
    earliest_dt = datetime.utcfromtimestamp(earliest / 1000)
    latest_dt = datetime.utcfromtimestamp(latest / 1000)
    print(f"{name}:")
    print(f"  Earliest timeMs: {earliest} ({earliest_dt})")
    print(f"  Latest   timeMs: {latest} ({latest_dt})\n")

show_df_time_range("options_df", options_df)

options_df:
  Earliest timeMs: 1756771200017 (2025-09-02 00:00:00.017000)
  Latest   timeMs: 1756857599990 (2025-09-02 23:59:59.990000)



/var/folders/fl/fdwpgpsx7fx15p__07t93bhr0000gn/T/ipykernel_28342/3855728042.py:12: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  earliest_dt = datetime.utcfromtimestamp(earliest / 1000)
/var/folders/fl/fdwpgpsx7fx15p__07t93bhr0000gn/T/ipykernel_28342/3855728042.py:13: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  latest_dt = datetime.utcfromtimestamp(latest / 1000)


In [9]:
bad_options_df = options_df.filter(
    (pl.col('ask_1_px') < pl.col('bid_1_px'))
)

bad_options_df.head()
print(bad_options_df.shape)
print(bad_options_df.head())


(2446, 11)
shape: (5, 11)
┌────────────┬───────────┬──────────┬───────────┬───┬──────────┬───────────┬───────────┬───────────┐
│ symbol     ┆ bid_1_px  ┆ ask_1_px ┆ timeMs    ┆ … ┆ opt_type ┆ F_bid     ┆ F_ask     ┆ moneyness │
│ ---        ┆ ---       ┆ ---      ┆ ---       ┆   ┆ ---      ┆ ---       ┆ ---       ┆ ---       │
│ str        ┆ f64       ┆ f64      ┆ i64       ┆   ┆ str      ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪═══════════╪══════════╪═══════════╪═══╪══════════╪═══════════╪═══════════╪═══════════╡
│ BTC-USD-25 ┆ 19.114961 ┆ 0.0      ┆ 175681920 ┆ … ┆ C        ┆ 109243.94 ┆ 109244.04 ┆ 0.951997  │
│ 0903-10400 ┆           ┆          ┆ 0000      ┆   ┆          ┆ 2066      ┆ 208       ┆           │
│ 0-C.OK     ┆           ┆          ┆           ┆   ┆          ┆           ┆           ┆           │
│ BTC-USD-25 ┆ 11.472883 ┆ 0.0      ┆ 175681950 ┆ … ┆ C        ┆ 109281.04 ┆ 109281.14 ┆ 0.951674  │
│ 0903-10400 ┆           ┆          ┆ 0000      ┆   ┆          ┆ 